# Chapter 11: Defect Analysis

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Defect Triage Assistant** that analyzes bug reports, classifies severity, identifies root cause patterns, suggests assignments, and detects duplicate defects.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Familiarity with defect lifecycle management


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Defect Classification and Enrichment

Take raw bug reports and automatically classify, prioritize, and enrich them with additional context.


In [ ]:
# Sample bug reports (as they might come from users or testers)
bug_reports = [
    {
        'id': 'BUG-301',
        'title': 'App crashes when uploading large file',
        'reported_by': 'tester_jane',
        'description': 'When I try to upload a PDF that is about 50MB, the app just crashes. No error message. It works fine for small files under 5MB. Tried on Chrome and Firefox, same result.',
        'environment': 'Production, Chrome 120, Windows 11'
    },
    {
        'id': 'BUG-302',
        'title': 'Wrong total on invoice',
        'reported_by': 'customer_support',
        'description': 'Customer reported their invoice shows $150 but they only ordered 2 items at $50 each. Happens when a discount code is applied and then removed. The discount seems to be subtracted but then added back incorrectly.',
        'environment': 'Production'
    },
    {
        'id': 'BUG-303',
        'title': 'Login button does not work sometimes',
        'reported_by': 'tester_bob',
        'description': 'Sometimes when I click the login button nothing happens. I have to click it 2-3 times. Not sure if its a timing issue. Only happens on slow connections.',
        'environment': 'Staging, Mobile Safari, iPhone 14'
    }
]

# Triage each bug report
def triage_bug(bug: dict) -> dict:
    prompt = f"""As a senior QA lead, triage this bug report. Analyze and provide:

Bug Report: {json.dumps(bug)}

Return a JSON object with:
- bug_id: original ID
- severity: Critical/High/Medium/Low (based on user impact and scope)
- priority: P1/P2/P3/P4 (based on business urgency)
- category: UI/API/Database/Performance/Security/Logic/Integration
- component: likely system component affected
- root_cause_hypothesis: most likely technical cause
- reproduction_steps: clear numbered steps to reproduce
- suggested_assignee_role: what type of developer should fix this
- additional_info_needed: questions to ask reporter for clarity
- workaround: temporary workaround if possible
- estimated_effort: hours estimate (S=1-2h, M=4-8h, L=16-24h, XL=40h+)

Return ONLY valid JSON."""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

triaged = []
for bug in bug_reports:
    result = triage_bug(bug)
    triaged.append(result)
    print(f"\n{result['bug_id']}: {bug['title']}")
    print(f"  Severity: {result['severity']} | Priority: {result['priority']}")
    print(f"  Category: {result['category']} | Component: {result['component']}")
    print(f"  Root Cause: {result['root_cause_hypothesis'][:80]}...")
    print(f"  Effort: {result['estimated_effort']}")

## 2. Duplicate Detection

Check if new bugs are duplicates of existing ones.


In [ ]:
# Existing bugs in the system
existing_bugs = [
    {'id': 'BUG-201', 'title': 'File upload fails for files over 25MB', 'status': 'Open'},
    {'id': 'BUG-155', 'title': 'Invoice calculation error with promotional codes', 'status': 'In Progress'},
    {'id': 'BUG-189', 'title': 'Button click not registering on mobile', 'status': 'Resolved'},
    {'id': 'BUG-210', 'title': 'Search results slow for large catalogs', 'status': 'Open'},
    {'id': 'BUG-175', 'title': 'Email notifications not sent for password reset', 'status': 'Closed'},
]

# Check for duplicates
dup_prompt = f"""Check if any of these new bug reports are duplicates or related to existing bugs.

New Bugs: {json.dumps(bug_reports)}
Existing Bugs: {json.dumps(existing_bugs)}

For each new bug, determine:
- is_duplicate: true/false
- related_bug_id: ID of the most similar existing bug (or null)
- similarity_score: 0-100 percentage
- relationship: 'duplicate', 'related', 'regression', or 'new'
- reasoning: why you think they are related or not

Return a JSON array. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': dup_prompt}],
    temperature=0.2
)
dup_results = json.loads(response.choices[0].message.content)
for d in dup_results:
    print(f"\n{d.get('bug_id', d.get('new_bug_id', 'N/A'))}: {d['relationship'].upper()}")
    if d.get('related_bug_id'):
        print(f"  Related to: {d['related_bug_id']} (similarity: {d['similarity_score']}%)")
    print(f"  Reasoning: {d['reasoning']}")

In [ ]:
# Trend analysis: identify patterns across bugs
trend_prompt = f"""Analyze these triaged bugs for patterns and trends.

Triaged Bugs: {json.dumps(triaged)}

Provide:
1. Common root cause patterns
2. Most affected components
3. Quality hotspots (areas needing more testing)
4. Process improvement recommendations
5. Testing gaps these bugs reveal

Format as a quality insights report with clear sections."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': trend_prompt}],
    temperature=0.4
)
print('QUALITY INSIGHTS REPORT')
print('=' * 60)
print(response.choices[0].message.content)

## Exercises
1. Build an auto-responder that generates initial responses to bug reporters with clarification questions.
2. Create a defect prediction model that identifies which features are likely to have more bugs.
3. Generate regression test cases from resolved bug reports.
4. Build a defect report quality scorer that rates how well a bug is documented.
